# 🚚 範例 09：智慧物流—宅配包裹延遲預警與調度 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「物流運輸主管、倉儲行控調度員與供應鏈管理人員（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以物流調度白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、物流資料集介紹 (`logistics.csv`)

本數據集記錄了宅配物流 2,000 件在途包裹的規格、轉運站狀態與最終配送時效結果：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 物流運輸業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **起運區域** | 類別型 (`object`) | 林口、台中、高雄轉運中心 | 發貨轉運起點 |
| **配送行政區** | 類別型 (`object`) | 台北市都會區、新北偏遠山區、花東線等 | 目的地地形與配送耗時差異 |
| **包裹重量_kg** | 數值型 (`float64`) | 8.5 (kg) | 貨物重量（影響搬運與車速） |
| **包裝材積大小** | 類別型 (`object`) | 小材積、中材積、大材積、超大材積 | 佔用貨艙體積 |
| **出貨時間帶** | 類別型 (`object`) | 晨間批次、午後批次、傍晚夜配批次 | 出發時段路況尖峰 |
| **當日轉運站包裹總量** | 數值型 (`int64`) | 68,000 (件) | 轉運站分揀與出車排隊壓力 |
| **降雨強度** | 類別型 (`object`) | 無雨/晴朗、局部陣雨、大雨滂沱/暴雨 | 行車安全速度限制 |
| **是否為大促檔期** | 類別型 (`object`) | 是、否 | 雙11電商大促爆倉指標 |
| **配送是否延遲 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：包裹是否逾時延遲妥投** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視行政區與雨勢之延遲分佈。
3. **特徵關聯性與視覺化**：繪製目的地與天候對延遲率之長條圖。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸」與「隨機森林分類器」。
6. **測試集大考與調度處遇**：評估 Recall 與混淆矩陣，輸出模型即時預估。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入物流時效分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並載入中文字型庫，確保路線圖與延遲率長條圖的文字正常渲染。

---
💡 **AI 提問小抄：**

1. 📌 **物流時效與機器學習**
   > 「傳統物流依賴老師傅憑經驗看天氣調派車輛。機器學習如何同時綜合考量『轉運站進件總量』、『材積大小』與『降雨強度』，提供更客觀的延遲機率評估？」


---  
### 【步驟 1】：載入資料與物流結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 logistics.csv 並檢視資料概況
df = pd.read_csv("logistics.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 延遲標籤分佈比例 ---")
print(df["配送是否延遲"].value_counts(normalize=True))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀取 2,000 件包裹數據，確認整體延遲率基準約為 30%。

---
💡 **AI 提問小抄：**

1. 📌 **延遲基準率之商業意涵**
   > 「30% 的延遲率在電商大檔期（如雙11）算常見嗎？如果平時常態妥投率要求達 95% 以上，這個資料集該如何配合月份進行動態校正？」


---  
### 【步驟 2】：行政區與降雨路況對延遲率之交叉圖表繪製

In [ ]:
# [步驟 2 程式碼] 繪製配送行政區與降雨強度的延遲率長條圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dest_order = df.groupby("配送行政區")["配送是否延遲"].apply(lambda s: (s == "是").mean()).sort_values(ascending=False)
sns.barplot(x=dest_order.index, y=dest_order.values * 100, palette="coolwarm", ax=axes[0])
axes[0].set_title("各配送行政區之包裹延遲率 (%)")
axes[0].set_ylabel("延遲率 (%)")
axes[0].tick_params(axis="x", rotation=25)

rain_order = df.groupby("降雨強度")["配送是否延遲"].apply(lambda s: (s == "是").mean())
sns.barplot(x=rain_order.index, y=rain_order.values * 100, palette="Blues_r", ax=axes[1])
axes[1].set_title("不同降雨強度之延遲率比較 (%)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  繪製行政區與降雨強度的延遲比例圖，突顯花東線與大雨滂沱對超時的強烈推動效果。

---
💡 **AI 提問小抄：**

1. 📌 **偏遠路線的地理障礙**
   > 「花東線與偏遠山區延遲率明顯高於都會區。在模型訓練時，我們該如何避免模型『對所有花東件都無腦猜延遲』？這需要加入哪些更細緻的特徵（如蘇花改即時路況）？」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["配送是否延遲"])
y = (df["配送是否延遲"] == "是").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ["包裹重量_kg", "當日轉運站包裹總量"]
cat_cols = ["起運區域", "配送行政區", "包裝材積大小", "出貨時間帶", "降雨強度", "是否為大促檔期"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集：{X_train.shape[0]} 筆，測試集：{X_test.shape[0]} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將數據拆分為 8:2，將重量與站所總量標準化，起迄站與天候進行獨熱編碼封裝。

---
💡 **AI 提問小抄：**

1. 📌 **防資料洩漏與推論一致性**
   > 「如果在特徵中包含了『司機實際出車時間』而不是『預定排班時段』，當出車本身就已經大塞車時，模型會如何被污染？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (邏輯回歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證比對邏輯回歸與隨機森林
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(random_state=42))])
pipe_rf = Pipeline([("preprocessor", preprocessor), ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="roc_auc")
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="roc_auc")

print(f"邏輯回歸 5 折交叉驗證 平均 ROC-AUC: {scores_lr.mean():.4f} (±{scores_lr.std():.4f})")
print(f"隨機森林 5 折交叉驗證 平均 ROC-AUC: {scores_rf.mean():.4f} (±{scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  運用 5 折交叉驗證比較兩者對轉運負載與天候複合瓶頸的預測表現。

---
💡 **AI 提問小抄：**

1. 📌 **隨機森林在供應鏈上的優勢**
   > 「為什麼隨機森林非常適合處理具有『木桶效應』（只要有一個環節如暴雨或站所爆倉出事，整條鏈路就超時）的供應鏈物流問題？」


---  
### 【步驟 5】：測試集大考與指標解讀 (混淆矩陣, Recall, ROC-AUC)

In [ ]:
# [步驟 5 程式碼] 在全新測試集進行評估並繪製混淆矩陣
from sklearn.metrics import classification_report, confusion_matrix

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)

print("=== 測試集最終評估報告 ===")
print(classification_report(y_test, y_pred, target_names=["準時(0)", "延遲(1)"]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["預測準時", "預測延遲"], yticklabels=["實際準時", "實際延遲"])
plt.title("物流包裹配送延遲預警混淆矩陣")
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  評估 400 件獨立測試包裹的混淆矩陣與 Recall 抓全率。

---
💡 **AI 提問小抄：**

1. 📌 **誤報 vs 漏報的成本比較**
   > 「在冷鏈生鮮食品配送中，漏報一個延遲件會導致整批食材腐壞賠償；而在一般常溫包裹中，誤報延遲可能只是發送一則提醒簡訊。行控中心該如何依據貨物價值設定不同的預警門檻？」


---  
### 【步驟 6】：在途包裹即時預警 (Inference) 與加派專車處遇

In [ ]:
# [步驟 6 程式碼] 保存模型管線並模擬在途包裹即時風險評估
joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬一件即將出車的急件包裹
urgent_package = pd.DataFrame([{
    "起運區域": "北部轉運中心(林口)",
    "配送行政區": "花蓮台東花東線",
    "包裹重量_kg": 12.0,
    "包裝材積大小": "大材積(91-120cm)",
    "出貨時間帶": "傍晚夜配批次(18:00)",
    "當日轉運站包裹總量": 72000,
    "降雨強度": "大雨滂沱/暴雨",
    "是否為大促檔期": "是"
}])

loaded_model = joblib.load("model.joblib")
delay_prob = loaded_model.predict_proba(urgent_package)[0][1]
print(f"\n🔔 該件包裹預估超時延遲機率：{delay_prob*100:.1f}%")
if delay_prob >= 0.6:
    print("🚨 極高延遲警報！受暴雨及花東偏遠路況影響，建議行控中心加派專車直達，並發送簡訊向收件人通報時效！")
else:
    print("🟢 正常件：維持常態幹線發車。")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  保存模型管線並展示行控中心即時輸入即將出庫的急件包裹，取得即時處遇建議。

---
💡 **AI 提問小抄：**

1. 📌 **即時串接與 GPS 整合**
   > 「在真實物流車隊管理系統 (TMS) 中，如何將這套模型與貨車車載 GPS 及即時 Google Traffic API 串接，每 15 分鐘重新推算全車包裹的妥投延遲風險？」
